In [1]:
import radiate as rd
import polars as pl
from IPython.display import display, HTML

rd.random.seed(67123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [3]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.linear()],
        edge=rd.Op.weight(),
        output=rd.Op.linear(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .alters(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-07-04T00:37:47.615537Z  INFO Epoch 1    | Score:   2.0038 | Time: 185.29µs
2026-07-04T00:37:47.615637Z  INFO Epoch 2    | Score:   2.0038 | Time: 242.12µs
2026-07-04T00:37:47.615704Z  INFO Epoch 3    | Score:   2.0038 | Time: 297.54µs
2026-07-04T00:37:47.615765Z  INFO Epoch 4    | Score:   1.6821 | Time: 347.25µs
2026-07-04T00:37:47.615826Z  INFO Epoch 5    | Score:   1.6821 | Time: 401.38µs
2026-07-04T00:37:47.615895Z  INFO Epoch 6    | Score:   1.6821 | Time: 462.21µs
2026-07-04T00:37:47.615958Z  INFO Epoch 7    | Score:   1.6821 | Time: 519.29µs
2026-07-04T00:37:47.616038Z  INFO Epoch 8    | Score:   1.6821 | Time: 590.62µs
2026-07-04T00:37:47.616104Z  INFO Epoch 9    | Score:   1.6821 | Time: 649.75µs
2026-07-04T00:37:47.616170Z  INFO Epoch 10   | Score:   1.6821 | Time: 708.38µs
2026-07-04T00:37:47.616228Z  INFO Epoch 11   | Score:   1.6821 | Time: 758.92µs
2026-07-04T00:37:47.616297Z  INFO Epoch 12   | Score:   1.6821 | Time: 821.71µs
2026-07-04T00:37:47.616353Z  INFO Epoch 

In [4]:
# collector.plot()
print(result.metrics().dashboard())

[37 metrics, 8608 updates]
----- Metrics ----- (37 :: 8608) 
  carryover: 0.379  diversity: 0.523  unique_members: 63  unique_scores: 52  improvements: 240  iter_time(mean): 165.504µs

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt      
---------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 0.990      | 0.000      | 4.000      | 100    | 0.099        | 1.219      | 4.909      | 27.853    
genome.size                | dist   | 20.070     | 20.000     | 21.000     | 100    | 2.007        | 0.256      | 0.000      | 0.000     
scores                     | dist   | 0.711      | 0.001      | 49.411     | 100    | 0.071        | 5.086      | 9.873      | 99.942    


== Statistics ==
Name                       | Type   | Mean       | Min        | Max        | N      | To

In [5]:
eval_results = result.value().eval(inputs)
accuracy = rd.accuracy(result.value(), inputs, answers, loss=rd.MSE)
accuracy

PyAccuracy("Regression Accuracy" {
	N: 20 
	Accuracy: 99.69%
	R² Score: 0.99976
	RMSE: 0.03127
	Loss (MSE): 0.00098
})

In [6]:
df = collector.to_polars(lazy=False)
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",8.0,108.0,54.0,65.053825,4232.0,NaN,8.0,100.0,2,null,null,null,null,null,null,0,2,"[""statistic""]"
"""step.evaluate.time""",0.000011,0.000077,0.000039,0.000039,1.4964e-9,NaN,0.000011,0.000066,2,77µs,38µs,38µs,11µs,65µs,0µs,0,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,20.0,20.0,0.0,0.0,NaN,20.0,20.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000006,0.000006,0.000006,0.0,0.0,NaN,0.000006,0.000006,1,6µs,6µs,0µs,6µs,6µs,0µs,0,1,"[""selector"", ""time""]"
"""selector.tournament""",80.0,80.0,80.0,0.0,0.0,NaN,80.0,80.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""scores.best""",0.000978,145.284042,0.60535,0.686918,0.471857,0.0,0.000978,2.003778,240,null,null,null,null,null,null,239,1,"[""statistic"", ""score""]"
"""step.metric.time""",0.000011,0.00265,0.000011,0.000003,6.9593e-12,0.0,0.000009,0.000041,240,2650µs,11µs,2µs,9µs,40µs,0µs,239,1,"[""time"", ""step""]"
"""time""",0.000261,0.039721,0.000166,0.00009,8.0936e-9,0.0,0.000042,0.000423,240,39720µs,165µs,89µs,42µs,423µs,0µs,239,1,"[""time""]"


In [7]:
filtered = (
    df.filter(pl.col("name") == "score.improvement")
    .select("generation")
    .unique()
    .sort("generation")
)
filtered

generation
i64
0
1
2
3
4
…
235
236
237


In [8]:
display(HTML(filtered._repr_html_()))

generation
i64
0
1
2
3
4
…
235
236
237
